# Emoji Prediction from Text: Data Preparation & Augmentation
This notebook processes the **TweetEval emoji dataset**:
- Groups similar emojis (hearts, cameras) into a single class.
- Keeps 11 relevant emoji classes.
- Re‑splits the data into train / validation / test.
- Augments minority classes using an external Kaggle dataset to balance the training set.
**Final output:** Balanced CSV files (`EPFT_train.csv`, `EPFT_val.csv`, `EPFT_test.csv`).

## 1. Imports and Configuration

In [1]:
import re
import html
import unicodedata
import numpy as np
import pandas as pd
import emoji as emoji_lib
from collections import Counter
from datasets import load_dataset
from sklearn.model_selection import train_test_split

CONFIG_NAME = "emoji"
RANDOM_SEED = 42

MAPPING = {
    "💕": "❤",
    "📷": "📷", "📸": "📷",
    "😂": "😂",
    "😎": "😎",
    "😉": "😉",
    "💯": "💯",
    "🔥": "🔥",
    "✨": "✨",
    "😊": "😊",
    "😜": "😜",
    "😁": "😁",
}

ALLOWED_CLASSES = set(MAPPING.values())

OUTPUT_TRAIN = "/kaggle/working/EPFT_train.csv"
OUTPUT_VAL   = "/kaggle/working/EPFT_val.csv"
OUTPUT_TEST  = "/kaggle/working/EPFT_test.csv"

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## 2. Load Original TweetEval Dataset & Explore Labels

In [2]:
print("=" * 60)
print("STEP 1: Loading TweetEval emoji dataset")
print("=" * 60)

dataset = load_dataset("cardiffnlp/tweet_eval", name=CONFIG_NAME)

label_names = dataset["train"].features["label"].names
num_labels = len(label_names)
print(f"Original labels ({num_labels}): {label_names}\n")

split_counts = {}
for split in ["train", "validation", "test"]:
    labels = dataset[split]["label"]
    unique, counts = np.unique(labels, return_counts=True)
    split_counts[split] = {label_names[i]: count for i, count in zip(unique, counts)}
    print(f"{split.upper()} total samples: {len(labels)}")

print("\n--- Original label distribution ---")
print(f"{'Label':<15} {'Train':>8} {'Val':>8} {'Test':>8}")
for label in label_names:
    train_c = split_counts["train"].get(label, 0)
    val_c   = split_counts["validation"].get(label, 0)
    test_c  = split_counts["test"].get(label, 0)
    print(f"{label:<15} {train_c:8d} {val_c:8d} {test_c:8d}")

STEP 1: Loading TweetEval emoji dataset


README.md: 0.00B [00:00, ?B/s]

emoji/train-00000-of-00001.parquet:   0%|          | 0.00/2.61M [00:00<?, ?B/s]

emoji/test-00000-of-00001.parquet:   0%|          | 0.00/3.05M [00:00<?, ?B/s]

emoji/validation-00000-of-00001.parquet:   0%|          | 0.00/282k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Original labels (20): ['❤', '😍', '😂', '💕', '🔥', '😊', '😎', '✨', '💙', '😘', '📷', '🇺🇸', '☀', '💜', '😉', '💯', '😁', '🎄', '📸', '😜']

TRAIN total samples: 45000
VALIDATION total samples: 5000
TEST total samples: 50000

--- Original label distribution ---
Label              Train      Val     Test
❤                   9204     1056    10798
😍                   4901      521     4830
😂                   4713      504     4534
💕                   2043      308     2605
🔥                   2146      243     3716
😊                   2132      238     1613
😎                   2078      204     1996
✨                   2345      199     2749
💙                   1287      177     1549
😘                   1391      171     1175
📷                   1982      159     1432
🇺🇸                   946      143     1949
☀                   1246      129     1265
💜                    980      153     1114
😉                   1224      129     1306
💯                    934      148     1244
😁                   135

## 3. Transform Labels & Keep Only Relevant Classes

In [3]:
print("\n" + "=" * 60)
print("STEP 2: Mapping emojis and filtering classes")
print("=" * 60)

idx_to_emoji = {idx: emoji for idx, emoji in enumerate(label_names)}

def transform_and_filter(split_data):
    """
    Convert original integer labels to new target labels using MAPPING.
    Discard any sample whose original emoji is not in MAPPING.
    Returns (texts, new_labels) for kept samples.
    """
    original_labels = split_data["label"]
    texts = split_data["text"]
    new_labels, kept_texts = [], []
    for lbl_int, text in zip(original_labels, texts):
        original_emoji = idx_to_emoji[lbl_int]
        if original_emoji in MAPPING:
            new_labels.append(MAPPING[original_emoji])
            kept_texts.append(text)
    return kept_texts, new_labels

train_texts, train_labels = transform_and_filter(dataset["train"])
test_texts, test_labels   = transform_and_filter(dataset["test"])
val_texts, val_labels     = transform_and_filter(dataset["validation"])

print(f"Original train: {len(dataset['train'])} -> kept: {len(train_texts)}")
print(f"Original test : {len(dataset['test'])}  -> kept: {len(test_texts)}")
print(f"Original valid : {len(dataset['validation'])} -> kept: {len(val_texts)}")


STEP 2: Mapping emojis and filtering classes
Original train: 45000 -> kept: 23648
Original test : 50000  -> kept: 25775
Original valid : 5000 -> kept: 2529


## 4. Create New Train / Validation / Test Splits
#
- Combine original **train + test** → new train (90%) + new validation (10%)
- Original **validation** → new test set

In [4]:
print("\n" + "=" * 60)
print("STEP 3: Creating new splits")
print("=" * 60)

all_train_texts = train_texts + test_texts
all_train_labels = train_labels + test_labels

new_train_texts, new_val_texts, new_train_labels, new_val_labels = train_test_split(
    all_train_texts, all_train_labels,
    test_size=0.1,
    random_state=RANDOM_SEED,
    stratify=all_train_labels
)

new_test_texts = val_texts
new_test_labels = val_labels

print(f"New train size:      {len(new_train_texts)}")
print(f"New validation size: {len(new_val_texts)}")
print(f"New test size:       {len(new_test_texts)}")

def print_distribution(labels, name):
    unique, counts = np.unique(labels, return_counts=True)
    print(f"\n{name} set class distribution:")
    for cls, cnt in zip(unique, counts):
        print(f"  {cls}: {cnt} ({cnt/len(labels):.2%})")

print_distribution(new_train_labels, "New train")
print_distribution(new_val_labels, "New validation")
print_distribution(new_test_labels, "New test")

pd.DataFrame({"text": new_train_texts, "label": new_train_labels}).to_csv(OUTPUT_TRAIN, index=False)
pd.DataFrame({"text": new_val_texts,   "label": new_val_labels}).to_csv(OUTPUT_VAL, index=False)
pd.DataFrame({"text": new_test_texts,  "label": new_test_labels}).to_csv(OUTPUT_TEST, index=False)
print(f"\nIntermediate splits saved to {OUTPUT_TRAIN}, {OUTPUT_VAL}, {OUTPUT_TEST}")


STEP 3: Creating new splits
New train size:      44480
New validation size: 4943
New test size:       2529

New train set class distribution:
  ✨: 4584 (10.31%)
  ❤: 4183 (9.40%)
  💯: 1960 (4.41%)
  📷: 6607 (14.85%)
  🔥: 5276 (11.86%)
  😁: 2253 (5.07%)
  😂: 8322 (18.71%)
  😉: 2277 (5.12%)
  😊: 3370 (7.58%)
  😎: 3667 (8.24%)
  😜: 1981 (4.45%)

New validation set class distribution:
  ✨: 510 (10.32%)
  ❤: 465 (9.41%)
  💯: 218 (4.41%)
  📷: 734 (14.85%)
  🔥: 586 (11.86%)
  😁: 250 (5.06%)
  😂: 925 (18.71%)
  😉: 253 (5.12%)
  😊: 375 (7.59%)
  😎: 407 (8.23%)
  😜: 220 (4.45%)

New test set class distribution:
  ✨: 199 (7.87%)
  ❤: 308 (12.18%)
  💯: 148 (5.85%)
  📷: 288 (11.39%)
  🔥: 243 (9.61%)
  😁: 137 (5.42%)
  😂: 504 (19.93%)
  😉: 129 (5.10%)
  😊: 238 (9.41%)
  😎: 204 (8.07%)
  😜: 131 (5.18%)

Intermediate splits saved to /kaggle/working/EPFT_train.csv, /kaggle/working/EPFT_val.csv, /kaggle/working/EPFT_test.csv


## 5. Augment Minority Classes with Kaggle Data
- This section uses an external Kaggle dataset (`Train.csv`, `Mapping.csv`) to balance the **training set**.
- The goal is to make all classes the same/close number of samples as the majority class.
- **Note:** The Kaggle dataset is **not** part of the original TweetEval.
- The code below assumes the Kaggle CSV files are located in:
`/kaggle/input/datasets/hariharasudhanas/twitter-emoji-prediction/`
Adjust the paths if needed.

In [5]:
print("\n" + "=" * 60)
print("STEP 4: Augmenting minority classes using Kaggle data")
print("=" * 60)

try:
    hf_train = pd.read_csv(OUTPUT_TRAIN)
    hf_train["label"] = hf_train["label"].astype(str)
    hf_counts = Counter(hf_train["label"])
    majority_count = max(hf_counts.values())
    
    print("Current class counts (before augmentation):")
    for emoji, cnt in sorted(hf_counts.items()):
        print(f"  {emoji}: {cnt}")
    print(f"Majority count: {majority_count}\n")
    
    kaggle_train = pd.read_csv("/kaggle/input/datasets/hariharasudhanas/twitter-emoji-prediction/Train.csv")
    kaggle_train = kaggle_train.rename(columns={"TEXT": "text", "Label": "label_num"})
    
    mapping = pd.read_csv("/kaggle/input/datasets/hariharasudhanas/twitter-emoji-prediction/Mapping.csv")
    id_to_emoji = dict(zip(mapping["number"], mapping["emoticons"]))
    
    kaggle_train["emoji"] = kaggle_train["label_num"].map(id_to_emoji)
    kaggle_filtered = kaggle_train[kaggle_train["emoji"].isin(ALLOWED_CLASSES)].copy()
    print(f"Kaggle: {len(kaggle_train)} rows → {len(kaggle_filtered)} usable rows")
    
    balanced_texts = hf_train["text"].tolist()
    balanced_labels = hf_train["label"].tolist()
    
    for emoji, current_cnt in hf_counts.items():
        needed = majority_count - current_cnt
        if needed <= 0:
            continue
        
        kaggle_class = kaggle_filtered[kaggle_filtered["emoji"] == emoji]
        available = len(kaggle_class)
        if available == 0:
            print(f"Warning: No Kaggle samples for class {emoji}. Cannot augment.")
            continue
        
        sample_size = min(needed, available)
        
        sampled = kaggle_class.sample(n=sample_size, random_state=RANDOM_SEED, replace=False)
        
        if available < needed:
            print(f"  Class {emoji}: needed {needed}, but only {available} available → added all {available} samples without replacement.")
        else:
            print(f"  Class {emoji}: added {sample_size} samples (needed {needed})")
        
        balanced_texts.extend(sampled["text"].tolist())
        balanced_labels.extend(sampled["emoji"].tolist())
    
    balanced_df = pd.DataFrame({"text": balanced_texts, "label": balanced_labels})
    new_counts = Counter(balanced_df["label"])
    
    print("\nClass counts AFTER augmentation:")
    for emoji, cnt in sorted(new_counts.items()):
        print(f"  {emoji}: {cnt} (target = {majority_count})")
        
    balanced_df = balanced_df.drop_duplicates(subset="text", keep="first")
    
    val_df = pd.read_csv(OUTPUT_VAL)
    test_df = pd.read_csv(OUTPUT_TEST)
    forbidden = set(val_df["text"]).union(set(test_df["text"]))
    balanced_df = balanced_df[~balanced_df["text"].isin(forbidden)]
    
    balanced_df.to_csv(OUTPUT_TRAIN, index=False)
    print(f"Final training set size after augmentation and deduplication: {len(balanced_df)}")
    
except FileNotFoundError as e:
    print(f"Kaggle data not found. Skipping augmentation. ({e})")
except Exception as e:
    print(f"An error occurred during augmentation: {e}")


STEP 4: Augmenting minority classes using Kaggle data
Current class counts (before augmentation):
  ✨: 4584
  ❤: 4183
  💯: 1960
  📷: 6607
  🔥: 5276
  😁: 2253
  😂: 8322
  😉: 2277
  😊: 3370
  😎: 3667
  😜: 1981
Majority count: 8322

Kaggle: 70000 rows → 44113 usable rows
  Class 📷: added 1715 samples (needed 1715)
  Class 😉: needed 6045, but only 1878 available → added all 1878 samples without replacement.
  Class 🔥: added 3046 samples (needed 3046)
  Class 😁: needed 6069, but only 1721 available → added all 1721 samples without replacement.
  Class 😜: needed 6341, but only 1557 available → added all 1557 samples without replacement.
  Class 😊: needed 4952, but only 2751 available → added all 2751 samples without replacement.
  Class 😎: needed 4655, but only 2832 available → added all 2832 samples without replacement.
  Class ❤: added 4139 samples (needed 4139)
  Class ✨: needed 3738, but only 3250 available → added all 3250 samples without replacement.
  Class 💯: needed 6362, but only 1

## 6. Final Dataset Cleaning & Overwriting
This section applies a cleaning function to our finalized train, val, and test files to ensure consistent data quality across all splits.

### Cleaning Strategy Justification:
- **Standardizing Handles/URLs:** Prevents the model from overfitting to specific users or websites.

- **Emoji Stripping:** Essential for prediction tasks. If the emoji remains in the text, the model learns a simple search task rather than understanding sentiment.

- **Unicode Normalization:** Removes "ghost" artifacts like Variation Selectors (\ufe0f) which often appear after emojis are stripped but act as meaningless noise.

- **Case Preservation:** We intentionally avoid lowercasing because capitalization (e.g., "SHOUTING") is a high-confidence signal for specific emojis like 🔥 or 😂.

In [6]:
print("\n" + "=" * 60)
print("STEP 5: Cleaning Pipeline")
print("=" * 60)

def text_clean(text):
    """
    Applies deep cleaning while preserving linguistic 'energy'.
    """
    if not isinstance(text, str):
        return ""
        
    text = re.sub(r"@[^\s]+", "@user", text)
    text = re.sub(r"http\S+|www\S+|https\S+", "http", text)
    
    text = emoji_lib.replace_emoji(text, replace='')
    
    text = "".join(ch for ch in text if unicodedata.category(ch)[0] != 'C')
    
    text = re.sub(r'\s+', ' ', text)
    
    text = re.sub(r'\s@\s\w+.*$', '', text) 

    text = html.unescape(text)
    
    return text.strip()

files_to_process = {
    "Train": OUTPUT_TRAIN,
    "Validation": OUTPUT_VAL,
    "Test": OUTPUT_TEST
}

print(f"{'Split':<12} | {'Original Size':<15} | {'Cleaned Size'}")
print("-" * 45)

for name, path in files_to_process.items():
    df = pd.read_csv(path)
    original_count = len(df)
    
    df["text"] = df["text"].apply(text_clean)
    
    df = df[df["text"].str.strip() != ""]
    df = df.drop_duplicates(subset="text")
    
    final_count = len(df)
    
    df.to_csv(path, index=False)
    
    print(f"{name:<12} | {original_count:<15} | {final_count}")

print("\nFinal clean datasets saved to directory. Ready for training.")

train_ds = pd.read_csv(OUTPUT_TRAIN)
train_ds


STEP 5: Cleaning Pipeline
Split        | Original Size   | Cleaned Size
---------------------------------------------
Train        | 68693           | 58498
Validation   | 4943            | 4922
Test         | 2529            | 2520

Final clean datasets saved to directory. Ready for training.


,text,label
0,Black & White : yayornayblog #videoshoot #reptilelover #bts #mcm,📷
1,Pancit! The belief that eating pancit on your bday will extend your life... #birthday…,😉
2,“Start a fire” they said. “You can’t screw it up” they said.,🔥
3,Ever since csimsw nicknamed him #kinghenry we’ve been dying to get a of this guy with a…,📷
4,Waiting patiently with my mini-me for @user while she's in…,😂
...,...,...
58493,For A Mf To Watch You Cry & Break Down Over Something & They Continue Doing The Same Shit They Do Not Love Or Care About You .. Period -Ny,💯
58494,"Goodluck at college cuz, love ya much",💯
58495,link in my profile #NewMusic #slightwork #trap #bass #MyStyle…,💯
58496,Raspberry Lime and a view above the crowd #Wurstfest #Wurstfest2016 #poptopiapops,💯
